In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\thaku\anaconda3\envs\llms_course_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\thaku\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Create training data

data = [
    ("Laptop screen is flickering", 0),
    ("Battery not charging", 0),

    ("Excel crashes on startup", 1),
    ("Unable to install software", 1),

    ("VPN not connecting", 2),
    ("Internet keeps dropping", 2),

    ("Forgot my password", 3),
    ("Cannot login to email", 3),
]

In [5]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": [x[0] for x in data],
    "label": [x[1] for x in data]
})

In [6]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length"
    )

dataset = dataset.map(tokenize)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [7]:
# Train the model

from transformers import Trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

  0%|          | 0/10 [00:00<?, ?it/s]

{'train_runtime': 40.1485, 'train_samples_per_second': 0.996, 'train_steps_per_second': 0.249, 'train_loss': 1.2546403884887696, 'epoch': 5.0}


TrainOutput(global_step=10, training_loss=1.2546403884887696, metrics={'train_runtime': 40.1485, 'train_samples_per_second': 0.996, 'train_steps_per_second': 0.249, 'total_flos': 5298884935680.0, 'train_loss': 1.2546403884887696, 'epoch': 5.0})

In [8]:
model.save_pretrained("ticket_classifier")
tokenizer.save_pretrained("ticket_classifier")

('ticket_classifier\\tokenizer_config.json',
 'ticket_classifier\\special_tokens_map.json',
 'ticket_classifier\\vocab.txt',
 'ticket_classifier\\added_tokens.json',
 'ticket_classifier\\tokenizer.json')

In [9]:
my_model = AutoModelForSequenceClassification.from_pretrained(
    "ticket_classifier"
)

my_tokenizer = AutoTokenizer.from_pretrained(
    "ticket_classifier"
)

In [18]:
text = "My VPN is not working. It's causing some issues."

inputs = my_tokenizer(
    text,
    return_tensors="pt"
)

outputs = my_model(**inputs)

prediction = outputs.logits.argmax().item()
print(prediction)

2


In [19]:
labels = {
    0: "Hardware",
    1: "Software",
    2: "Network",
    3: "Account Access"
}

print(labels[prediction])

Network
